In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.integrate import odeint
from scipy.fftpack import diff as psdiff

from mpl_toolkits.mplot3d import Axes3D

---
# 1. Introduction
The Lorenz63 system is known as a chaotical system which may not be well modelled by neural networks. However, its characteristic of chaos exists inherently in many atmospheric and oceanic systems. Machine Learning methods have been widely applied into earth system modelling over the past decades. Especially, many AI-based big models (K. Bi, et al. 2023) claimed that they had achieved superior forecast skills than traditional numerical models. So a 'paradox' arises: if the AI algorithms cannot learn a complexity-reduced model, why they can learn the much more complex mechanisms in the atmosphere? To improve the understanding of AI-driven weather forecasting, we will examine the AI-predictability of lorenz63 system by adopting basic RNN algorithm.

Many previous researches have shown that the neural networks should be designed specifically in order to provide accurate predictions (Wang X., et al. 2024). Since the neural networks adopted here are general and basic, we don't expect the predictions close to the true data. Instead, we want to test whether the RNNs can learn the statistical and geometric properties of lorenz63 system, such as its attractor structure.

In [ ]:
def lorenz63(t, state, sigma, rho, beta):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

state0 = [1.0, 1.0, 1.0]  # Initial state
params = (10.0, 28.0, 8.0/3.0)  # Classial arameters for sigma, rho, beta

t = np.linspace(0, 100, 10000)
sol = solve_ivp(lorenz63, [t[0], t[-1]], [0.1, 0.1, 0.1], t_eval=t, args=params)

x, y, z = sol.y
x = x[1000:]
y = y[1000:]
z = z[1000:]

In [ ]:
def create_sequences(data, seq_length=1):
    x_dum, y_dum = [], []
    n_sample = data.shape[0] - seq_length
    for i in range(n_sample):
        x_dum.append(data[i:i+seq_length, :])  # sequence used to predict end point
        y_dum.append(data[i+seq_length, :])    # end point
    return np.asarray(x_dum), np.asarray(y_dum)

data = np.stack([x, y, z], axis=0)

X, Y = create_sequences(data.T, seq_length=10)

X_train, Y_train = X, Y

print(f"input shape is (batch_size, seq, dim) = {X.shape}")
print(f"input shape is (batch_size, dim) = {Y.shape}")

---
# 2. Methods

## 2.1 Models

We choose two RNN models to predict lorenz63 system. The first model is a simple RNN, which only contains one hiden state.The second model is LSTM, which contains forget, input and output gates, more complex than the former one. As a result, under similar structures the parameter size of LSTM is much larger than that of the simple RNN.

In order to leverage the adventage of RNNs, we set the input sequence length as ten. In other words, the model utilizes the data from the previous ten time steps as input to predict the system state at the subsequent step. One-step prediction is not tested here because it performs poorly in other simpler tasks and do not make full use of the recurrent structure of RNNs. Besides,since lorenz63 system is hard to learn, the training data are normalized before provided to the RNNs. In order to make the two models comparable, the models are set to have similar degrees of freedom.  





In [ ]:
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

# keras related
import os
os.environ["KERAS_BACKEND"] = "torch"  # use PyTorch as backend
import keras
import keras.layers as layers
keras.backend.clear_session() # force a clean keras session (clears models etc.)

from tqdm.keras import TqdmCallback

from keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [ ]:
class MyDataset(Dataset):

    def __init__(self, in_tensor, out_tensor):
        self.inp = in_tensor
        self.out = out_tensor

    def __len__(self):
        return len(self.inp)

    def __getitem__(self, idx):
        return self.inp[idx], self.out[idx]

In [ ]:
def simple_rnn(input_size, output_size, seq_length=1, hidden_size=1, num_layers=1):

    norm_layer = layers.Normalization(axis=-1)
    norm_layer.adapt(X_train)

    # need to use the "Cell" variant to loop up
    rnn_cells = [layers.SimpleRNNCell(hidden_size) for _ in range(num_layers)]

    inputs = keras.Input(shape=(seq_length, input_size))
    x = norm_layer(inputs)
    x = layers.RNN(rnn_cells)(x)  # this is one block
    outputs = layers.Dense(output_size)(x)
    model = keras.Model(inputs, outputs, name="simple RNN")

    return model

RNN = simple_rnn(3, 3, seq_length=10, hidden_size=120, num_layers=1)
RNN.summary()

In [ ]:
def simple_lstm(input_size, output_size, seq_length=10, hidden_size=1, num_layers=1):

    norm_layer = layers.Normalization(axis=-1)
    norm_layer.adapt(X_train)

    # need to use the "Cell" variant to loop up
    lstm_cells = [layers.LSTMCell(hidden_size) for _ in range(num_layers)]

    inputs = keras.Input(shape=(seq_length, input_size))
    x = norm_layer(inputs)
    x = layers.RNN(lstm_cells)(x)  # this is one block
    outputs = layers.Dense(output_size)(x)
    model = keras.Model(inputs, outputs, name="simple LSTM")

    return model

LSTM = simple_lstm(3, 3, seq_length=10, hidden_size=60, num_layers=1)
LSTM.summary()

In [ ]:
torch.manual_seed(1234)
keras.utils.set_random_seed(4321)

# use all the data for training (don't have to do this)
train_dataset = MyDataset(X, Y)
n_sample = train_dataset.__len__()

train_dataloader = DataLoader(train_dataset,
                              batch_size=n_sample // 1,
                              shuffle=True)


Training on lorenz63 dataset is slow, so we adopt the 'early stop' to accelerate the training progress. This strategy allow the model to be optimized fast at first and be trained carefully when the model state is around the extreme point.

In [ ]:
# Use early stop to adjust learning rate

learning_rate = 0.1

reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=100,
    min_lr=0.01,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='loss',
    patience=2000,
    restore_best_weights=True,
    verbose=1
)

---
## 2.2 Metrics

To evauate the performance of these models, some metrics should be introduced. One simple qualitative way is to plot the trajectory in the 3D phase space and observe whether it looks like a 'butterfly'. This criterion can identify the ineffective model but is difficult to compare two effective models quantitatively. Therefore, we introduce two metrics to test the statistical and geometric properties respectively.

### 2.2.1 Wasserstein Distance

Wasserstein Distance determines the difference between two probability distributions. Its idea is to find the optimal transport from one distribution to another distribution (Villani C. 2009). Take two gaussian distributions as examples. If the expectation values of these two distributions are the same, the wasserstain distance between them is

\begin{equation}
W = \sqrt{2/\pi} |\sigma_1 - \sigma_2|.
\end{equation}

If the standard deviations of two gaussian distributions are the same, the wasserstein distance is

\begin{equation}
W = |\mu_1 - \mu_2|.
\end{equation}

For 1D distributions, the numerical calculation of Wasserstein distance is easy because we can use the function `wasserstein_distance` in `SciPy` package. However, lorenz63 system has three variables and thus its probabilty distribution is a 3D joint distribution. To simplify the problem, we only consider the marginal distributions of variables $x$ and $y$.

The follow figure shows the marginal distributions of $x$ and $y$ in lorenz63 system and calculate the wasserstein distance bewteen them. The physical meaning of this calculation is not very clear and we only regard it as an example. The figure demonstrates that the distributions of $x$ and $y$ are both symmetric about zero, which implies the numbers of point in two atractors are approxmately equal. This statistical fact may be related to the inherent symmetry in lorenz63 system, where if $(x,y,z)$ is the solution, $(-x,-y,z)$ must be the solution.  In this case, the symmetry acutally depends on the initial condition because the time series are not long enough.

In [ ]:
import seaborn as sns
from scipy.stats import wasserstein_distance

wd_xy = wasserstein_distance(x[750:], y[750:])

sns.kdeplot(x[750:], fill=True, label="X", color="red", alpha=0.5)
sns.kdeplot(y[750:], fill=True, label="Y", color="royalblue", alpha=0.5)
plt.title(f'Marginal PDFs of X and Y\n wd = {wd_xy:4f}')
plt.grid()
plt.legend()

### 2.2.2 Fractal Dimension

The trajectory of lorenz63 system seems to lie on a 2D plane embedded in the 3D space. Previous studies have shown that the fractal dimension of lorenz63 system approximately equals to 2.06, very close to our intuition (Aizawa Y., 1982).

There are many ways to calculate the fractal dimension numerically. The method we adopted here is the correlation dimension (Nerenberg M A H, et al. 1990). This method tries to find the relationship between the radius $r$ of a ball and the number of points in this ball, denoted by $C(r)$. Ideally, a 2D surface in 3D space obsesses such relationship (based on the formula for the area of a circle),

\begin{equation}
C(r) \propto r^2.
\end{equation}

Still considering 2D smooth manifolds, we only define the smooth structure locally, so such dimension is accurate only when $r$ is very small. In practice, there is also a distance between each two points. We cannot set $r$ so small that the ball contains no points. Take the logarithm of both sides of the above equation,

\begin{equation}
\log C(r) = 2 \cdot \log r + \mathrm{const}.
\end{equation}

Therefore, we only need to fit the data $(\log C(r),\log r)$ and the dimension is the slope. The following function is written by gemini to realize this idea. The numerical result of 2.06 is perfectly in line with the previous studies.


In [ ]:
from scipy.spatial.distance import pdist

def calculate_correlation_dimension(data, min_r_factor=0.002, max_r_factor=0.5, num_r_points=50, fit_start_percent=0.25, fit_end_percent=0.75):
    """
    Calculates the correlation dimension of a given dataset using the correlation sum method.
    Assumes input data is always valid and sufficient for calculation.
    """
    N = data.shape[0]
    distances = pdist(data)
    max_dist = np.max(distances)
    min_r = max_dist * min_r_factor
    max_r = max_dist * max_r_factor

    r_values = np.logspace(np.log10(min_r), np.log10(max_r), num_r_points)

    correlation_sums = np.array([(2.0 / (N * (N - 1))) * np.sum(distances < r) for r in r_values])

    # Filter for values amenable to log-log plot and linear fit
    valid_indices = (correlation_sums > 0) & (correlation_sums < 1.0)
    log_r = np.log(r_values[valid_indices])
    log_cr = np.log(correlation_sums[valid_indices])

    # Select a region for linear regression (central part of the log-log plot)
    start_idx = int(len(log_r) * fit_start_percent)
    end_idx = int(len(log_r) * fit_end_percent)

    log_r_fit = log_r[start_idx:end_idx + 1]
    log_cr_fit = log_cr[start_idx:end_idx + 1]

    coeffs = np.polyfit(log_r_fit, log_cr_fit, 1)
    fractal_dimension = coeffs[0]

    log_cr_fit_line = coeffs[0] * log_r_fit + coeffs[1]

    return r_values, correlation_sums, fractal_dimension, log_r_fit, log_cr_fit_line

In [ ]:
lorenz_data = np.stack([x,y,z], axis=1)

r_lorenz, C_r_lorenz, dim_lorenz, log_r_fit_lorenz, log_cr_fit_line_lorenz = calculate_correlation_dimension(lorenz_data)

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.plot(log_r_fit_lorenz, log_cr_fit_line_lorenz, '--', color='red', label=f'D={dim_lorenz:.2f}')
ax.scatter(log_r_fit_lorenz, log_cr_fit_line_lorenz)
ax.set_xlabel('log(r)')
ax.set_ylabel('log(C(r))')
ax.set_title('Correlation Dimension')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

# 3. Results

Each training session lasts about 5 minutes on GPU T4 and the outcomes may vary slightly. We generate the prediction by using RNNs autoregressively, i.e.

\begin{equation}
\hat x_{t+10} = \mathrm{RNNs}(\hat x_{t+9},...,\hat x_{t})
\end{equation}

Without running testing codes, we can first compare the loss function curve. The curve of the LSTM is very smooth, except for the sections where the learning rate is changed, while the curve of the RNN fluctuates greatly. Moreover, the training losses of the RNN is much larger than the one of LSTM. These two properties both implies that the RNN may perform worse than the LSTM in this task.

In [ ]:
RNN.compile(loss=keras.losses.MeanSquaredError(),
             optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
             )

train_rnn = RNN.fit(train_dataloader,
                     epochs=2000 ,
                     verbose=0,
                     callbacks=[TqdmCallback(verbose=1),reduce_lr,early_stop],
                    )

# plot the loss curves
fig = plt.figure(figsize=(6, 3))
ax = plt.axes()
ax.plot(train_rnn.epoch, train_rnn.history["loss"], label="training loss")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel(r"$J$")
ax.grid()
ax.legend();

In [ ]:
learning_rate = 0.1

reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=100,
    min_lr=0.01,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='loss',
    patience=2000,
    restore_best_weights=True,
    verbose=1
)

LSTM.compile(loss=keras.losses.MeanSquaredError(),
             optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
             )

train_lstm = LSTM.fit(train_dataloader,
                     epochs=2000 ,
                     verbose=0,
                     callbacks=[TqdmCallback(verbose=1),reduce_lr,early_stop],
                    )

# plot the loss curves
fig = plt.figure(figsize=(6, 3))
ax = plt.axes()
ax.plot(train_lstm.epoch, train_lstm.history["loss"], label="training loss")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel(r"$J$")
ax.grid()
ax.legend();

(Each test takes approximately 2 minutes.)

In [ ]:
initial_condition = 750
predict_length = 7000

X_in = X_train[initial_condition, :, :]

RNNpredictions_from_init = np.zeros(Y_train[:predict_length-initial_condition,:].shape)

for i in range(predict_length-initial_condition):
    Y_pred = RNN.predict(X_in.reshape(-1, 10, 3), verbose=0)  # suppress outputs
    RNNpredictions_from_init[i, :] = Y_pred.squeeze() # squeeze that dummy singleton dim

    # update the input sequence
    X_dum = np.zeros(X_in.shape)
    X_dum[0:-1, :] = X_in[1:, :]
    X_dum[-1, :] = Y_pred
    X_in = X_dum

In [ ]:
X_in = X_train[initial_condition, :, :]

LSTMpredictions_from_init = np.zeros(Y_train[:predict_length-initial_condition,:].shape)

for i in range(predict_length-initial_condition):
    Y_pred = LSTM.predict(X_in.reshape(-1, 10, 3), verbose=0)  # suppress outputs
    LSTMpredictions_from_init[i, :] = Y_pred.squeeze() # squeeze that dummy singleton dim

    # update the input sequence
    X_dum = np.zeros(X_in.shape)
    X_dum[0:-1, :] = X_in[1:, :]
    X_dum[-1, :] = Y_pred
    X_in = X_dum

In [ ]:
X_lstm = LSTMpredictions_from_init[:,0]
Y_lstm = LSTMpredictions_from_init[:,1]
Z_lstm = LSTMpredictions_from_init[:,2]

X_rnn = RNNpredictions_from_init[:,0]
Y_rnn = RNNpredictions_from_init[:,1]
Z_rnn = RNNpredictions_from_init[:,2]

The prediction of the RNN is irregular and finally converges to a stable point (or caught in high-frequency oscillation). Thus we regard a simple RNN as an ineffective model for lorenz63 system. By contrast, the LSTM can mimic the behavior of lorenz63 system, although the time series begins to diverge from the ground truth after several hundreds steps.

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax1 = plt.subplot(3, 1, 1)
ax1.plot(x[initial_condition:predict_length], '-', label='x')
ax1.plot(X_rnn, label='X_rnn')
ax1.grid()
ax1.legend()

ax2 = plt.subplot(3, 1, 2)
ax2.plot(y[initial_condition:predict_length], '-', label='y')
ax2.plot(Y_rnn, label='Y_rnn')
ax2.grid()
ax2.legend()

ax3 = plt.subplot(3, 1, 3)
ax3.plot(z[initial_condition:predict_length], '-', label='z')
ax3.plot(Z_rnn, label='Z_rnn')
ax3.set_xlabel('time steps')
ax3.grid()
ax3.legend()

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax1 = plt.subplot(3, 1, 1)
ax1.plot(x[initial_condition:predict_length], '-', label='x')
ax1.plot(X_lstm, label='X_lstm')
ax1.grid()
ax1.legend()

ax2 = plt.subplot(3, 1, 2)
ax2.plot(y[initial_condition:predict_length], '-', label='y')
ax2.plot(Y_lstm, label='Y_lstm')
ax2.grid()
ax2.legend()

ax3 = plt.subplot(3, 1, 3)
ax3.plot(z[initial_condition:predict_length], '-', label='z')
ax3.plot(Z_lstm, label='Z_lstm')
ax3.set_xlabel('time steps')
ax3.grid()
ax3.legend()

The trajectory of LSTM prediction also looks like a 'butterfly' except for some subtle fluctuations.

In [ ]:
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot(x[:predict_length], y[:predict_length], z[:predict_length], label='Numerical Model', color='blue', lw=0.5)
ax1.set_title('The Lorenz63 System')
ax1.set_xlabel(r"$x(t)$")
ax1.set_ylabel(r"$y(t)$")
ax1.set_zlabel(r"$z(t)$")

ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax2.plot(X_rnn, Y_rnn, Z_rnn, label='RNN', color='red', lw=0.5)
ax2.set_title('RNN trajectory')
ax2.set_xlabel(r"$x(t)$")
ax2.set_ylabel(r"$y(t)$")
ax2.set_zlabel(r"$z(t)$")

ax3 = fig.add_subplot(1, 3, 3, projection='3d')
ax3.plot(X_lstm, Y_lstm, Z_lstm, label='LSTM', color='green', lw=0.5)
ax3.set_title('LSTM trajectory')
ax3.set_xlabel(r"$x(t)$")
ax3.set_ylabel(r"$y(t)$")
ax3.set_zlabel(r"$z(t)$")

plt.tight_layout()
plt.show()

The following figure shows the PDFs of the three models. The PDF of RNN has a peak in a small section. The PDF of the LSTM prediction is close to the true PDF. Minor deviations are observed across different training sessions. The PDFs are asymmetric in most cases and the LSTM tends to stay at the neighborhood of zero. In any case, the LSTM maintains a low Wasserstein distance relative to the true PDF.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

wd_lstm = wasserstein_distance(x[750:], X_lstm)
wd_rnn = wasserstein_distance(x[750:], X_rnn)

sns.kdeplot(x[initial_condition:predict_length], fill=True, label="Target", color="red", alpha=0.5, ax=ax1)
sns.kdeplot(X_rnn, fill=True, label=f"RNN (Wd:{wd_rnn:.4f})", color="green", alpha=0.5, ax=ax1)
sns.kdeplot(X_lstm, fill=True, label=f"LSTM (Wd:{wd_lstm:.4f})", color="royalblue", alpha=0.5, ax=ax1)

ax1.set_ylim([0, 0.08])
ax1.set_title("X PDFs")
ax1.legend()

wd_lstm = wasserstein_distance(y[750:], Y_lstm)
wd_rnn = wasserstein_distance(y[750:], Y_rnn)


sns.kdeplot(y[initial_condition:predict_length], fill=True, label="Target", color="red", alpha=0.5, ax=ax2)
sns.kdeplot(Y_rnn, fill=True, label=f"RNN (Wd:{wd_rnn:.4f})", color="green", alpha=0.5, ax=ax2)
sns.kdeplot(Y_lstm, fill=True, label=f"LSTM (Wd:{wd_lstm:.4f})", color="royalblue", alpha=0.5, ax=ax2)

ax2.set_ylim([0, 0.08])
ax2.set_title("Y PDFs")
ax2.legend()


plt.tight_layout()
plt.show()

The fractal dimension of the RNN prediction is very small because it behaves like a line or a point. The dimension of the LSTM prediction is a bit less than 2.06. This result demonstrates that the LSTM can partly recover the structure of lorenz63 system, but its prediction is closer to a 2D plane that the model truth. One possible explanation for this small deviation may be its relatively poor skills in predicting the transition area between two attractors.

In [ ]:
lorenz_data = np.stack([x,y,z], axis=1)

r_lorenz, C_r_lorenz, dim_lorenz, log_r_fit_lorenz, log_cr_fit_line_lorenz = calculate_correlation_dimension(lorenz_data)
r_lstm, C_r_lstm, dim_lstm, log_r_fit_lstm, log_cr_fit_line_lstm = calculate_correlation_dimension(LSTMpredictions_from_init)
r_rnn, C_r_rnn, dim_rnn, log_r_fit_rnn, log_cr_fit_line_rnn = calculate_correlation_dimension(RNNpredictions_from_init)

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.plot(log_r_fit_lorenz, log_cr_fit_line_lorenz, '--', color='blue', label=f'Lorenz Fit (D={dim_lorenz:.2f})')
ax.plot(log_r_fit_lstm, log_cr_fit_line_lstm, '--', color='red', label=f'LSTM Predicted Fit (D={dim_lstm:.2f})')
ax.plot(log_r_fit_rnn, log_cr_fit_line_rnn, '--', color='green', label=f'RNN Predicted Fit (D={dim_rnn:.2f})')

ax.set_xlabel('log(r)')
ax.set_ylabel('log(C(r))')
ax.set_title('Correlation Dimension')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

# 4. Conclusion

In this assignment, we employ two neural networks to predict lorenz63 system. The autoregressive tests demonstrate that the two models both diverge from the model truth. However, the LSTM model has the potential to reconstruct the geometric shape of lorenz63 system, while the simple RNN cannot learn the its structure effectively. Additional evaluations using Wasserstein distance and fractal dimension indicate that the LSTM can effectively produce the fundamental statistical and geometric characteristics of the Lorenz-63 attractor. The reasons of small deviations need to be examined further.

# Reference:

Wang X, Feng J, Xu Y, et al. Deep learning-based state prediction of the Lorenz system with control parameters[J]. Chaos: An Interdisciplinary Journal of Nonlinear Science, 2024, 34(3).

Bi K, Xie L, Zhang H, et al. Accurate medium-range global weather forecasting with 3D neural networks[J]. Nature, 2023, 619(7970): 533-538.

Aizawa Y. Global Aspects of the Dissipative Dynamical Systems. I: —Statistical Identification and Fractal Properties of the Lorenz Chaos—[J]. Progress of Theoretical Physics, 1982, 68(1): 64-84.

Villani C. The wasserstein distances[M] Optimal transport: old and new. Berlin, Heidelberg: Springer Berlin Heidelberg, 2009: 93-111.

Nerenberg M A H, Essex C. Correlation dimension and systematic geometric effects[J]. Physical Review A, 1990, 42(12): 7065.